# 🎯 QLoRA Finetune + Eval Lab
### Tasar'u · NVIDIA Platform & Cert Prep — hands-on lab

Finetuning a full 7B updates ~14 GB of weights — impossible on a free T4. **QLoRA** makes it
fit: freeze the model in **4-bit**, and train only tiny **LoRA** adapters (a few million
params). Then we **evaluate** properly — because "it trained" is not "it got better."

| Step | What you do | Maps to (slides) |
|---|---|---|
| 1–2 | Load a 4-bit base, attach LoRA adapters | QLoRA / PEFT · memory sharding |
| 3–4 | Finetune on a small instruction set | SFT |
| 5 | **Eval base vs tuned** — perplexity + behavior | evaluation metrics |
| 6 | Scale up: **FSDP across 2×T4** | multi-GPU finetuning |

**Platform:** Colab / Kaggle single **T4** for steps 1–5. Step 6 needs Kaggle **2×T4**.


## 0 · Setup

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" bitsandbytes accelerate datasets 2>/dev/null
import torch, math, time
print("GPU:", torch.cuda.get_device_name(0))

## 1 · Load the base model in 4-bit
We use a small base so the lab finishes in minutes. **Swap `BASE` for
`Qwen/Qwen2.5-7B-Instruct` to do the real 7B QLoRA — identical code, just slower.**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
BASE = "Qwen/Qwen2.5-1.5B"      # -> "Qwen/Qwen2.5-7B-Instruct" for the full 7B run
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
print("4-bit base loaded:", f"{torch.cuda.memory_allocated()/1e9:.2f} GB of weights")

### Hold-out eval passage & a perplexity function
We measure **perplexity** (lower = the model is less "surprised" by the text) on a fixed
held-out passage, *before* any training.

In [ ]:
EVAL_TEXT = ("### Instruction:\nExplain why tensor parallelism must stay inside a GPU node.\n"
             "### Response:\nTensor parallelism performs an all-reduce every layer, so it needs the "
             "fast NVLink fabric inside a node; sending it across the slower InfiniBand between nodes "
             "would stall training.")
def perplexity(model, tok, text):
    ids = tok(text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad(): loss = model(ids, labels=ids).loss
    return math.exp(loss.item())
ppl_before = perplexity(model, tok, EVAL_TEXT)
print(f"Perplexity BEFORE finetuning: {ppl_before:.2f}")

## 2 · Attach LoRA adapters (the only trainable weights)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM", target_modules="all-linear")
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # ~ <1% of params are trainable

## 3 · A small instruction dataset
Alpaca — a classic instruction set. We take a small slice and format each example as
`### Instruction … ### Response …`, then tokenize.

In [ ]:
from datasets import load_dataset
raw = load_dataset("tatsu-lab/alpaca", split="train[:800]")
def fmt(ex):
    instr = ex["instruction"] + (("\n" + ex["input"]) if ex["input"] else "")
    return {"text": f"### Instruction:\n{instr}\n### Response:\n{ex['output']}{tok.eos_token}"}
ds = raw.map(fmt, remove_columns=raw.column_names)
def tokize(ex): return tok(ex["text"], truncation=True, max_length=256)
ds = ds.map(tokize, remove_columns=["text"])
print(ds, "\nexample tokens:", len(ds[0]["input_ids"]))

## 4 · Finetune (a short run so it completes)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tok, mlm=False)
args = TrainingArguments(
    output_dir="qlora-out", per_device_train_batch_size=2, gradient_accumulation_steps=4,
    max_steps=60, learning_rate=2e-4, fp16=True, logging_steps=10,
    gradient_checkpointing=True, report_to="none", save_strategy="no")
model.config.use_cache = False
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=collator)
trainer.train()
print("done — trained only the LoRA adapters on a 4-bit frozen base.")

## 5 · Evaluate: did it actually get better?
Two lenses, straight from the metrics slide:
1. **Perplexity** on the held-out passage — before vs after (quantitative).
2. **Behavior** — generate on a fresh instruction and eyeball the format/quality (qualitative).

In [ ]:
model.config.use_cache = True
ppl_after = perplexity(model, tok, EVAL_TEXT)
print(f"Perplexity  BEFORE: {ppl_before:.2f}   AFTER: {ppl_after:.2f}   "
      f"({'improved ✅' if ppl_after < ppl_before else 'no improvement ⚠️'})")

In [ ]:
# Qualitative: does it follow the instruction format?
prompt = "### Instruction:\nList two benefits of GPUDirect Storage.\n### Response:\n"
ids = tok(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**ids, max_new_tokens=80, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))

### Why perplexity alone isn't enough
Perplexity measures *fluency on this text*, not *task correctness*. For a real project you'd
add a **task metric** — accuracy / F1 for classification, exact-match for extraction, or a
rubric/LLM-judge for open generation. Perplexity is a cheap first signal, not the verdict.

## 6 · Scale it up — FSDP finetune across 2×T4  *(Kaggle)*
QLoRA on one GPU trains adapters. To **full-finetune** a model too big for one GPU, you shard
its parameters/gradients/optimizer across GPUs with **FSDP** — the "split the memory" lesson.
You already have a launch-ready script (`Finetuning_Lab/finetune.py` / repo `06_finetuning`).
On Kaggle 2×T4 you'd launch it like this:
```bash
accelerate launch --multi_gpu --num_processes 2 \
  finetune.py --model Qwen/Qwen2.5-1.5B --strategy fsdp --dataset alpaca
```

In [ ]:
import torch
if torch.cuda.device_count() < 2:
    print("Single GPU here — QLoRA (steps 1–5) is the right tool. For FSDP full-finetune, "
          "use Kaggle 'GPU T4 ×2' and the accelerate launch command above.")
else:
    print(f"{torch.cuda.device_count()} GPUs available — you can run the FSDP script across both. "
          "FSDP shards params/grads/optimizer so each GPU holds only its slice.")

## 7 · Reflection
1. QLoRA trained <1% of the parameters. Where did the memory savings come from — the **4-bit
   base**, the **small adapters**, or both? Which one let the model *fit*?
2. Your perplexity moved. Design one **task metric** that would better prove the finetune
   helped for a real use case.
3. QLoRA (adapters, one GPU) vs FSDP full-finetune (all weights, many GPUs): when would you
   reach for each? What does each *cost*?
